# 07 Comparison and interpretation

Filled in together after each model notebook has been merged.
This notebook only reads `results/*_metrics.json`. It does not retrain anything.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import common

EXPECTED = ["dt", "rf", "xgb", "logreg", "mlp"]
found = {row["model"]: row for row in common.load_all_metrics()}
missing = [name for name in EXPECTED if name not in found]
print("missing metrics:", missing or "none")

rows = []
for name in EXPECTED:
    row = found.get(name)
    if row is None:
        continue
    holdout = row["holdout"]
    rows.append({
        "model": name,
        "cv_roc_auc": row["cv"]["roc_auc_mean"],
        "cv_roc_auc_std": row["cv"]["roc_auc_std"],
        "holdout_roc_auc": holdout["roc_auc"],
        "holdout_pr_auc": holdout["pr_auc"],
        "precision": holdout["precision"],
        "recall": holdout["recall"],
        "f1": holdout["f1"],
        "accuracy": holdout["accuracy"],
        "seed": row["seed"],
        "test_size": row["test_size"],
    })

comparison = pd.DataFrame(rows)
comparison


## Interpretation

Write this section together, from the table above.

- Which model has the highest holdout ROC-AUC, and is the gap larger than the CV standard deviation?
- Precision and recall matter here because only about 13% of customers default. Accuracy alone will look strong for a model that almost never predicts default.
- What does the decision tree show that the forest and XGBoost hide?
- Which logistic-regression coefficients match the Component 1 story (DTI, late payments, text risk)?
- Did the MLP beat logistic regression by enough to justify the extra complexity?
